In [1]:
tools = [
    {
        "type": "function",
        "name": "calculate",
        "description": "Multiply two numbers.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {
                    "type": "number",
                    "description": "First number"
                },
                "b": {
                    "type": "number",
                    "description": "Second number"
                }
            },
            "required": ["a", "b"]
        }
    }
]

In [2]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-5.6",
    input="What is 25 multiplied by 48?",
    tools=tools
)

print(response)

Response(id='resp_061cde4ed607cce8006a81c744f0d087d0875c355fafb189ff', created_at=1786890052.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.6-sol', object='response', output=[ResponseFunctionToolCall(arguments='{"a":25,"b":48}', call_id='call_F9OwhPQO1sv5a6l4pepYDl9K', name='calculate', type='function_call', id='fc_061cde4ed607cce8006a81c7464ff887d09b20354f95fa6ac3', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='calculate', parameters={'type': 'object', 'properties': {'a': {'type': 'number', 'description': 'First number'}, 'b': {'type': 'number', 'description': 'Second number'}}, 'required': ['a', 'b'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Multiply two numbers.', output_schema=None)], top_p=0.98, background=False, completed_at=1786890054.0, conversation=None, max_output_tokens=None, max_tool_calls=None, previous

In [5]:
tool_call = response.output[0]

print(tool_call.name)
# calculate

print(tool_call.arguments)
# {"a":25,"b":48}

print(tool_call.call_id)
# call_F9OwhPQO1sv5a6l4pepYDl9K...

calculate
{"a":25,"b":48}
call_F9OwhPQO1sv5a6l4pepYDl9K


In [6]:
import json

args = json.loads(tool_call.arguments)

print(args["a"])
# 25

print(args["b"])
# 48

25
48


In [7]:
def calculate(a, b):
    return a * b

result = calculate(args["a"], args["b"])

print(result)
# 1200

1200


In [8]:
response = client.responses.create(
    model="gpt-5.6",
    previous_response_id=response.id,
    input=[
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": str(result)
        }
    ]
)

print(response)

Response(id='resp_061cde4ed607cce8006a81c874728087d0bace47ac022ffcbb', created_at=1786890356.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.6-sol', object='response', output=[ResponseOutputMessage(id='msg_061cde4ed607cce8006a81c875b65487d0b41d4daca08eac1d', content=[ResponseOutputText(annotations=[], text='25 × 48 = **1,200**.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1786890358.0, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id='resp_061cde4ed607cce8006a81c744f0d087d0875c355fafb189ff', prompt=None, prompt_cache_key=None, prompt_cache_retention='24h', reasoning=Reasoning(effort='medium', generate_summary=None, summary=None, context='all_turns', mode='standard'), safety_identifier=None, service_tier='default', status='completed

In [9]:
import json
from openai import OpenAI

client = OpenAI()

tools = [
    {
        "type": "function",
        "name": "calculate",
        "description": "Multiply two numbers.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {
                    "type": "number",
                    "description": "First number"
                },
                "b": {
                    "type": "number",
                    "description": "Second number"
                }
            },
            "required": ["a", "b"]
        }
    }
]

def calculate(a, b):
    return a * b


# 1. Ask the model
response = client.responses.create(
    model="gpt-5.6",
    input="What is 25 multiplied by 48?",
    tools=tools
)

# 2. Look for tool calls
for item in response.output:
    if item.type == "function_call":
        # 3. Extract arguments
        args = json.loads(item.arguments)

        # 4. Execute your actual function
        result = calculate(args["a"], args["b"])

        # 5. Send result back
        response = client.responses.create(
            model="gpt-5.6",
            previous_response_id=response.id,
            input=[
                {
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": str(result)
                }
            ]
        )

# 6. Extract the model's final text
print(response.output_text)

25 × 48 = **1,200**.


In [10]:
import json
from openai import OpenAI

client = OpenAI()

tools = [
    {
        "type": "function",
        "name": "calculate",
        "description": "Multiply two numbers.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {
                    "type": "number",
                    "description": "First number"
                },
                "b": {
                    "type": "number",
                    "description": "Second number"
                }
            },
            "required": ["a", "b"]
        }
    }
]


def calculate(a, b):
    return a * b


print("STEP 1: Sending user request to model")
print("User: What is 25 multiplied by 48?\n")


# 1. Ask the model
response = client.responses.create(
    model="gpt-5.6",
    input="What is 25 multiplied by 48?",
    tools=tools,
    reasoning={
        "effort": "medium",
        "summary": "auto"
    }
)

print("STEP 2: Model response received")


# Inspect the model's output items
for item in response.output:

    print("\n--- OUTPUT ITEM ---")
    print("Type:", item.type)

    # Reasoning summary, when returned
    if item.type == "reasoning":
        print("Reasoning summary:")

        for summary in item.summary:
            print(summary.text)

    # Function call
    elif item.type == "function_call":
        print("Model wants to call:", item.name)
        print("Arguments:", item.arguments)
        print("Call ID:", item.call_id)


        # 3. Parse arguments
        args = json.loads(item.arguments)

        print("\nSTEP 3: Parsed tool arguments")
        print("a =", args["a"])
        print("b =", args["b"])


        # 4. Execute your function
        print("\nSTEP 4: Executing Python function")

        result = calculate(args["a"], args["b"])

        print(
            f"calculate({args['a']}, {args['b']}) = {result}"
        )


        # 5. Send result back to model
        print("\nSTEP 5: Sending tool result back to model")

        response = client.responses.create(
            model="gpt-5.6",
            previous_response_id=response.id,
            input=[
                {
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": str(result)
                }
            ],
            reasoning={
                "effort": "medium",
                "summary": "auto"
            }
        )


# 6. Final answer
print("\nSTEP 6: Final model answer")
print(response.output_text)

STEP 1: Sending user request to model
User: What is 25 multiplied by 48?

STEP 2: Model response received

--- OUTPUT ITEM ---
Type: function_call
Model wants to call: calculate
Arguments: {"a":25,"b":48}
Call ID: call_llBEj8Gc51IRvLQQYyK4QEef

STEP 3: Parsed tool arguments
a = 25
b = 48

STEP 4: Executing Python function
calculate(25, 48) = 1200

STEP 5: Sending tool result back to model

STEP 6: Final model answer
25 × 48 = **1,200**.
